# 任意 - Travel Ops Toolbox の SDK 操作

本編の [Lab 4](../labs/04-tools-toolbox.md) は Portal で Toolbox、OpenAPI tool、Skills を追加します。この Notebook は SDK の学習・UI が利用できない場合の補助用で、本編の必須手順ではありません。

この Notebook は Travel Ops OpenAPI、Code Interpreter、Web Search、Tool Search と接続を作成し、Prompt Agent を呼び出して出力と Conversation の内部処理を確認します。Skills の新規アップロードは本編の Portal 手順で行ってください。既存 Toolbox を更新する場合、UI で追加した Skills、tool の個別設定、guardrail、Tool Search は保持します。Skill の自動利用を保証する Notebook ではなく、`load_skill` または `resources/read` がなければ利用済みとは扱いません。

**使用する kernel:** `Python (Foundry Workshop)`

Lab 1 で準備した Azure ML Studio の User files から、この Notebook を開きます。

認証には `az login` のセッションだけを使います。API key や client secret は読み込みません。上から順に 1 cell ずつ実行してください。

## 1. Repository root を確認する

Notebook をどのフォルダーから開いても、`.workshop/context.json` と既存の Python module を見つけられるようにします。

In [ ]:
import sys
from pathlib import Path


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(
        "Repository root が見つかりません。Lab 1 で準備した教材内の Notebook を開いてください。"
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository: {REPO_ROOT.name}")

## 2. Workshop context を読み込む

参加者ごとに異なる endpoint や resource 名は、Lab 1 が生成した `.workshop/context.json` から取得します。値を Notebook に貼り付ける必要はありません。

In [ ]:
from scripts.lib.workshop_context import (
    build_credential,
    load_context,
    project_endpoint,
    travel_api_base_url,
    workshop_output,
)

CONTEXT_PATH = REPO_ROOT / ".workshop" / "context.json"
TOOLBOX_NAME = "contoso-travel-toolbox"
TOOL_NAME = "travel_ops_api"
CONNECTION_NAME = "contoso-travel-toolbox-mcp"
AGENT_NAME = "contoso-travel-assistant"

context = load_context(CONTEXT_PATH)
foundry_endpoint = project_endpoint(context)
project_resource_id = workshop_output(context, "foundry_project_id")
travel_api_url = travel_api_base_url(context)

print(f"Foundry project endpoint: {foundry_endpoint}")
print(f"Travel Ops API: {travel_api_url}")

## 3. Foundry client を作成する

`AzureCliCredential` は、Lab 1 で実行した `az login` のキャッシュを使います。

In [ ]:
from azure.ai.projects import AIProjectClient

credential = build_credential("azure-cli")
client = AIProjectClient(endpoint=foundry_endpoint, credential=credential, allow_preview=True)

print("Foundry client を作成しました。")

## 4. Travel Ops API の OpenAPI 定義を取得する

実際にデプロイされた API の `/openapi.json` を取得します。Container App が停止中でも、既存 helper が有限回だけ再試行します。

In [ ]:
from scripts import create_toolbox

openapi_spec = create_toolbox.fetch_openapi_spec(
    travel_api_url,
    create_toolbox.DEFAULT_OPENAPI_PATH,
)
operation_ids = [
    operation["operationId"]
    for path in openapi_spec["paths"].values()
    for operation in path.values()
    if isinstance(operation, dict) and "operationId" in operation
]

print(f"OpenAPI: {openapi_spec['openapi']}")
print("Operations:", ", ".join(operation_ids))

## 5. OpenAPI tool を定義する

Travel Ops API は合成データだけを返す公開 mock API なので、認証方式は `anonymous` です。実運用 API では、この部分を managed identity などへ置き換えます。

In [ ]:
auth = create_toolbox.build_auth_details("anonymous", audience=None)
travel_ops_tool = create_toolbox.build_openapi_tool(
    tool_name=TOOL_NAME,
    spec=openapi_spec,
    auth=auth,
    description=(
        "Contoso の日当照会、費用見積もり、事前承認シミュレーションを実行する Travel Ops API。"
    ),
)

print(f"Tool: {travel_ops_tool.name}")

## 6. Toolbox を作成または更新する

同じ構成で再実行した場合は既存 Toolbox を再利用します。定義が変わった場合だけ新しい version を作り、default に設定します。既存の Portal 設定は保持し、不足する Lab 4 の built-in tool と Tool Search だけを追加します。参加者が version 番号を管理する必要はありません。

In [ ]:
result = create_toolbox.ensure_toolbox(
    client,
    endpoint=foundry_endpoint,
    toolbox_name=TOOLBOX_NAME,
    desired_tool=travel_ops_tool,
)

print(f"Action: {result['action']}")
print(f"Toolbox: {result['toolbox_name']}")
toolbox_endpoint = result["endpoints"]["consumer"]
print(f"MCP endpoint: {toolbox_endpoint}")

## 7. 作成結果を確認する

default の Toolbox に `travel_ops_api`、Code Interpreter、Web Search、Tool Search が含まれることを SDK で確認します。OpenAPI 1 項目は `getHealth` / `getPerDiem` / `createTripEstimate` / `createPreapproval` を公開します。Skills をまだアップロードしていない場合は、本編の Portal 手順へ戻って追加してください。

In [ ]:
toolbox = client.toolboxes.get(TOOLBOX_NAME)
toolbox_version = client.toolboxes.get_version(TOOLBOX_NAME, toolbox.default_version)
tool_names = [tool.name for tool in toolbox_version.tools]
tool_types = [tool.type for tool in toolbox_version.tools]

assert TOOL_NAME in tool_names, f"{TOOL_NAME} が Toolbox にありません: {tool_names}"
assert {"code_interpreter", "web_search", "toolbox_search"} <= set(tool_types)
print("Toolbox names:", ", ".join(name or "(unnamed)" for name in tool_names))
print("Toolbox types:", ", ".join(tool_types))

## 8. Prompt Agent 用の keyless connection を作成する

Toolbox は MCP endpoint として公開されます。Prompt Agent から secret なしで呼び出せるよう、project managed identity を使う connection を作成します。

In [ ]:
connection = create_toolbox.ensure_toolbox_connection(
    credential=credential,
    project_resource_id=project_resource_id,
    connection_name=CONNECTION_NAME,
    toolbox_endpoint=toolbox_endpoint,
)

print(f"Connection ready: {connection['name']}")

## 9. Toolbox を Prompt Agent に接続する

現在の agent に接続済みの Foundry IQ knowledge base は残したまま、Toolbox MCP tool を追加します。同じ構成で再実行した場合は既存の接続を再利用します。

In [ ]:
agent_result = create_toolbox.attach_toolbox_to_agent(
    client,
    agent_name=AGENT_NAME,
    connection_name=CONNECTION_NAME,
    toolbox_endpoint=toolbox_endpoint,
)

print(f"Agent: {agent_result['agent_name']}")
print(f"Version: {agent_result['agent_version']}")
print(f"Action: {agent_result['action']}")
print("Toolbox connection ready (tool calls are auto-approved).")

## 10. Prompt Agent を呼び出して出力を確認する

Agent 専用の OpenAI client と新しい Conversation を作成し、Lab 4 と同じ見積もり依頼を送信します。Section 9 で Toolbox MCP tool を自動承認にしているため、確認待ちで停止せず最終回答まで取得します。この呼び出しではモデル利用料金が発生します。

In [ ]:
REQUEST = """東京からニューヨークへ2026-07-10〜2026-07-15の出張で、
1名、ビジネスクラス利用を前提に費用見積もりを出してください。
予約や承認シミュレーションは不要です。"""

openai_client = client.get_openai_client(agent_name=AGENT_NAME)
conversation = openai_client.conversations.create()
response = openai_client.responses.create(
    conversation=conversation.id,
    input=REQUEST,
)
answer = response.output_text.strip()
normalized_answer = answer.replace(",", "")

assert answer, "Agent から空の回答が返されました。"
assert "781000" in normalized_answer, f"期待する合計 781,000 円がありません:\n{answer}"
assert "予約" in answer and "承認" in answer, "予約・承認ではない旨が回答にありません。"

print(f"Conversation ID: {conversation.id}")
print(f"Response ID: {response.id}")
print("\nAgent output:\n")
print(answer)

## 11. Conversation の内部処理を SDK で確認する

Responses API で同じ Conversation の items を取得し、Agent が記録した処理順を表示します。Tool Search では `tool_search` と `call_tool` を確認し、選択された実 tool と引数・出力を別に確認します。runtime によって downstream call は `sample.tool_calls` 相当へ flatten されず、`call_tool` の内側だけに現れるため、固定の item shape は仮定しません。

これは Conversation に保存された処理履歴です。各処理の所要時間などを調べる Application Insights の Trace とは異なります。

In [ ]:
import json

conversation_items = openai_client.conversations.items.list(
    conversation.id,
    order="asc",
).data
item_payloads = [item.model_dump(mode="json") for item in conversation_items]

print("Conversation processing items:\n")
for index, payload in enumerate(item_payloads, start=1):
    item_type = payload.get("type", "unknown")
    item_name = payload.get("name") or "-"
    item_status = payload.get("status") or "-"
    print(f"{index}. type={item_type}, name={item_name}, status={item_status}")

mcp_calls = [payload for payload in item_payloads if payload.get("type") == "mcp_call"]
call_names = [call.get("name") for call in mcp_calls]
assert "tool_search" in call_names, f"tool_search がありません: {call_names}"
assert "call_tool" in call_names, f"call_tool がありません: {call_names}"
assert all(call.get("status") == "completed" for call in mcp_calls)
assert all(call.get("error") is None for call in mcp_calls)

selected_calls = [call for call in mcp_calls if call.get("name") == "call_tool"]
selected_arguments = [json.loads(call["arguments"]) for call in selected_calls]
selected_json = json.dumps(selected_arguments, ensure_ascii=False)
assert "createTripEstimate" in selected_json, selected_json
irrelevant_tools = (
    "createPreapproval",
    "getHealth",
    "getPerDiem",
    "web_search",
    "code_interpreter",
)
for irrelevant in irrelevant_tools:
    assert irrelevant not in selected_json, f"不要な tool が実行されました: {irrelevant}"
for expected in ("Tokyo", "New York", "2026-07-10", "2026-07-15", "business"):
    assert expected in selected_json, f"call_tool 引数に {expected} がありません"

selected_outputs = [call.get("output", "") for call in selected_calls]
output_json = json.dumps(selected_outputs, ensure_ascii=False)
assert "total_estimate" in output_json, output_json
assert "781000" in output_json, output_json

print("\nTool Search order:", " -> ".join(call_names))
print("\nSelected actual tool and arguments:")
print(json.dumps(selected_arguments, ensure_ascii=False, indent=2))
print("\ncall_tool output:")
print(json.dumps(selected_outputs, ensure_ascii=False, indent=2))

## 12. Client を終了する

確認が完了したら、Notebook で使用した SDK client と credential を閉じます。作成した Conversation と Trace は Portal からも引き続き確認できます。

In [ ]:
openai_client.close()
client.close()
credential.close()

print("Clients closed.")